# DBT Wrapper

DBT Wrapper that runs DBT in stages, while sending monitoring logs in between.

As the Monitoring library is in Scala, it requires calling Scala code from Python.

#### Parameters

* **dbt_project**: Path to the the DBT project.
* **model_paths**: Paths to the models to run in order. Defaults to ["silver", "gold"].
* **target**: Target profile to use in DBT.
* **properties_file**: Path to the properties file containing the Monitoring Exporter properties.
* **job_name**: Name of the job for the monitoring logs.
* **job_id**: Id of the job for the monitoring logs.
* **run_id**: Id of the run for the monitoring logs.
* **extra_flags**: Map of extra flags that may be needed for DBT run where the keys must be values of model_paths. Any value of model_paths not present in the map will run without any extra flags. Defaults to {}.
* **tags**: Map of tags to send on the log. Defaults to {}.

### Install DBT

Installs the required DBT libraries and re-starts python to allow for their use.

In [0]:
%pip install dbt-core dbt-databricks databricks-sql-connector

In [0]:
dbutils.library.restartPython()

### Set-Up DBT

Sets up the DBT models to run and the DBT project directory based on the Notebook parameters 

In [0]:
import json
import os
from datetime import datetime
from zoneinfo import ZoneInfo

# Get DBT models to run
dbutils.widgets.text("model_paths", "[]")
model_paths = json.loads(dbutils.widgets.get("model_paths"))

if len(model_paths) == 0:
    model_paths = ["silver", "gold"]

dbutils.widgets.text("extra_flags", "{}")
extra_flags = json.loads(dbutils.widgets.get("extra_flags"))

dbutils.widgets.text("tags", "{}")
tags = json.loads(dbutils.widgets.get("tags"))

timestamp = datetime.now(ZoneInfo("Europe/Lisbon")).strftime("%Y%m%d_%H%M%S")

# Set-up DBT environmental variables
os.environ["DBT_PROJECT_DIR"] = dbutils.widgets.get("dbt_project")
os.environ["DBT_PROFILES_DIR"] = dbutils.widgets.get("dbt_project")
os.environ["DBT_LOG_PATH"] = f"{dbutils.widgets.get("dbt_project")}/logs/{timestamp}_{dbutils.widgets.get("run_id")}"

### Set-Up the Monitoring Exporter

Initializes the Monitoring Exporter and loads the Scala classes necessary to use it with Python

In [0]:
%scala
import com.brisa.monitoring.MonitoringExporter
import com.brisa.monitoring.model.{Log, LogStatusType}

val propertiesFile = dbutils.widgets.get("properties_file")
MonitoringExporter.initialize(propertiesFile, "DBT", dbutils.widgets.get("job_name"), dbutils.widgets.get("job_id"), dbutils.widgets.get("run_id"), None, None)

In [0]:
jvm = spark._jvm
HashMap = jvm.scala.collection.mutable.HashMap
LocalDateTime = jvm.java.time.LocalDateTime
ZoneOffset = jvm.java.time.ZoneOffset
Log = jvm.com.brisa.monitoring.model.Log
LogStatusType = jvm.com.brisa.monitoring.model.LogStatusType
MonitoringExporter = jvm.com.brisa.monitoring.MonitoringExporter

### Utils

Util functions

In [0]:
from dbt.cli.main import dbtRunnerResult

def create_successful_models_metrics(runner_result: dbtRunnerResult):
    """
    Creates the metrics for the successful models, namely the elapsed time in seconds.
    """
    metrics = HashMap()
    for model_result in runner_result.result.results:
        if model_result.status.value == "success":
            time = HashMap()
            time.put("elapsed", model_result.execution_time)
            metrics.put(model_result.node.name, time)
    return metrics

def create_successful_metrics(runner_result: dbtRunnerResult):
    """
    Creates the metrics for a successful DBT run
    """
    metrics = HashMap()
    successful_models = create_successful_models_metrics(runner_result)
    metrics.put("successful_models", successful_models)
    return metrics

def create_failure_metrics(runner_result: dbtRunnerResult):
    """
    Creates the metrics for a failed DBT run
    """
    metrics = HashMap()
    if runner_result.result:
        successful_models = create_successful_models_metrics(runner_result)
        metrics.put("successful_models", successful_models)

        failed_models = [model_result.node.name for model_result in runner_result.result.results if model_result.status.value != "success"]
        metrics.put("failed_models", failed_models)

        errors = HashMap()
        for model_result in runner_result.result.results:
            if model_result.status.value != "success":
                if model_result.message:
                    errors.put(model_result.node.name, model_result.message[:200])

        metrics.put("errors", errors)
    elif runner_result.exception:
        metrics.put("errors", str(runner_result.exception)[:1000])
    else:
        metrics.put("errors", "Unknown errors")

    return metrics

### Main

Set-up required variables and run DBT once per model_path, sending a monitoring log before and after each run.

In [0]:
from dbt.cli.main import dbtRunner

tool = "DBT"
process = dbutils.widgets.get("job_name")
job_id = dbutils.widgets.get("job_id")
run_id = dbutils.widgets.get("run_id")
target = dbutils.widgets.get("target")
bronze_time_column = dbutils.widgets.get("bronze_time_column")
bronze_path = dbutils.widgets.get("bronze_path")
silver_time_column = dbutils.widgets.get("silver_time_column")
silver_path = dbutils.widgets.get("silver_path")
extra_info = HashMap()

execution_arguments = HashMap()
execution_arguments.put("properties_file", dbutils.widgets.get("properties_file"))
execution_arguments.put("model_paths", model_paths)
execution_arguments.put("target", target)
execution_arguments.put("dbt_project", dbutils.widgets.get("dbt_project"))
execution_arguments.put("extra_flags", extra_flags)

scala_tags = HashMap()
for key, value in tags.items():
    scala_tags.put(key, value)

dbt_runner = dbtRunner()

In [0]:
from pyspark.sql.utils import AnalysisException
bronze_time_column = dbutils.widgets.get("bronze_time_column")
bronze_path = dbutils.widgets.get("bronze_path")
silver_time_column = dbutils.widgets.get("silver_time_column")
silver_path = dbutils.widgets.get("silver_path")
# Query to retrieve the list of execution dates that have not been processed
try:
    query = f"""
        SELECT CAST({bronze_time_column} AS STRING) FROM (
            SELECT DISTINCT {bronze_time_column}
            FROM {bronze_path}
            WHERE {bronze_time_column} > Coalesce(
                (SELECT MAX({silver_time_column}) FROM {silver_path}),
                '1900-01-01 00:00:00.000'
            )
        )
        ORDER BY {bronze_time_column} ASC
    """
    executions_in_order = spark.sql(query)
except AnalysisException as e:
    if "[TABLE_OR_VIEW_NOT_FOUND]" in str(e):
        query = f"""
            SELECT CAST({bronze_time_column} AS STRING) FROM (
                SELECT DISTINCT {bronze_time_column}
                FROM {bronze_path}
            )
            ORDER BY {bronze_time_column} ASC
        """
        executions_in_order = spark.sql(query)
    else:
        raise

execution_dates = [
    execution[bronze_time_column]
    for execution in executions_in_order.collect()
]

In [0]:
for date in execution_dates:
    print(date)

In [0]:
# Function to add the date flag to the extra flags
def add_date_flag(model_path, extra_flags, date):
    if model_path in extra_flags:
        if "vars" in extra_flags[model_path]:
            extra_flags[model_path]["vars"]["exec_date"] = date
        else:
            extra_flags[model_path]["vars"] = {"exec_date": date}
    else:
        extra_flags[model_path] = {"vars": {"exec_date": date}}

In [0]:
for model_path in model_paths:
    metrics = HashMap()
    starting_log = Log(tool, process, job_id, run_id, execution_arguments, scala_tags, model_path, LogStatusType.RUNNING, metrics, extra_info, f"Running models from '{model_path}'", LocalDateTime.now(ZoneOffset.UTC))
    MonitoringExporter.sendLog(starting_log)
    for date in execution_dates:
        stage = model_path + "_" + date
        metrics = HashMap()
        running_log = Log(tool, process, job_id, run_id, execution_arguments, scala_tags, stage, LogStatusType.RUNNING, metrics, extra_info, f"Running models from '{model_path}' for date '{date}'", LocalDateTime.now(ZoneOffset.UTC))
        MonitoringExporter.sendLog(running_log)
        try:
            commands = ["run", "--target", target, "--select", model_path]
            add_date_flag(model_path, extra_flags, date)
            if model_path in extra_flags:
                flags = extra_flags[model_path]
                for key, value in flags.items():
                    commands.append(f"--{key}")
                    if isinstance(value, (dict, list)):
                        commands.append(json.dumps(value))
                    elif value != "":
                        commands.append(str(value))
            print(commands)
            result = dbt_runner.invoke(commands)
            
            if result.success:
                status = LogStatusType.SUCCESSFUL
                message = f"Ran models from '{model_path}' successfully"
                metrics = create_successful_metrics(result)
            else:
                status = LogStatusType.FAILED
                message = f"Failed to run models from '{model_path}'"
                metrics = create_failure_metrics(result)
                
            log = Log(tool, process, job_id, run_id, execution_arguments, scala_tags, stage, status, metrics, extra_info, message, LocalDateTime.now(ZoneOffset.UTC))
            MonitoringExporter.sendLog(log)

        except Exception as exception:
            status = LogStatusType.FAILED
            message = f"Unexpected failure when running models from '{model_path}' and date '{date}'"
            metrics.put("errors", str(exception)[:1000])

            log = Log(tool, process, job_id, run_id, execution_arguments, scala_tags, stage, status, metrics, extra_info, message, LocalDateTime.now(ZoneOffset.UTC))
            MonitoringExporter.sendLog(log)

            metrics = HashMap()
            finishing_log = Log(tool, process, job_id, run_id, execution_arguments, scala_tags, model_path, status, metrics, extra_info, f"Failure in execution of model '{model_path}'.", LocalDateTime.now(ZoneOffset.UTC))
            MonitoringExporter.sendLog(finishing_log)

            raise exception

        if status == LogStatusType.FAILED:
            raise Exception(f"Failed during '{model_path}' models")

    metrics = HashMap()
    number_of_dates = len(execution_dates) - 1
    finishing_log = Log(tool, process, job_id, run_id, execution_arguments, scala_tags, model_path, LogStatusType.SUCCESSFUL, metrics, extra_info, f"For model '{model_path}', a total of '{number_of_dates}' dates were executed.", LocalDateTime.now(ZoneOffset.UTC))
    MonitoringExporter.sendLog(finishing_log)
    
    
